In [ ]:
# ============================================================
# PatchCore EfficientNet-B5 Layer Combination Comparison
# Method: PatchCore + Greedy Coreset + Max Image Score
# Added: Local /content dataset copy + compute/end-to-end timing
# Dataset: Lusitano_Dataset
#
# Output columns:
# Backbone | Layer_Combination | Layer_Type | D | Backbone_MB |
# Memory_Bank_MB | Total_MB | Peak_GPU_Memory_MB |
# Time/Image | AUC | AP | F1 | Edge_Efficiency | Edge_Efficiency_%
#
# SEED = 42 | No CenterCrop
# ============================================================

# ============================================================
# 1) Imports
# ============================================================

import os
import gc
import time
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b5, EfficientNet_B5_Weights
from PIL import Image, ImageFile

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

from google.colab import drive
drive.mount("/content/drive")

# ============================================================
# 2) Reproducibility
# ============================================================

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

ImageFile.LOAD_TRUNCATED_IMAGES = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

try:
    torch.set_float32_matmul_precision("medium")
except Exception:
    pass

# ============================================================
# 3) Dataset paths: copy Google Drive dataset to local /content
# ============================================================
# Why local copy?
#   Reading thousands of images directly from Google Drive can make
#   inference time look much slower than the actual model computation.
#   This block keeps the original dataset in Drive, copies it once to
#   local Colab storage, then uses /content/Lusitano_Dataset for the run.

COPY_DATASET_TO_LOCAL = True

DRIVE_DATASET_ROOT = Path("/content/drive/MyDrive/<YOUR_DATASET_FOLDER>/Lusitano_Dataset")
LOCAL_DATASET_ROOT = Path("/content/Lusitano_Dataset")

if COPY_DATASET_TO_LOCAL:
    if not DRIVE_DATASET_ROOT.exists():
        raise ValueError(f"Drive dataset not found: {DRIVE_DATASET_ROOT}")

    expected_local_train = LOCAL_DATASET_ROOT / "nondefects" / "nondefects"
    expected_local_test = LOCAL_DATASET_ROOT / "test" / "test"

    if expected_local_train.exists() and expected_local_test.exists():
        print("Local dataset already exists:", LOCAL_DATASET_ROOT)
    else:
        if LOCAL_DATASET_ROOT.exists():
            print("Removing incomplete local dataset copy...")
            shutil.rmtree(LOCAL_DATASET_ROOT)

        print("Copying dataset from Google Drive to local Colab storage...")
        print("Source:", DRIVE_DATASET_ROOT)
        print("Target:", LOCAL_DATASET_ROOT)
        copy_start = time.time()
        shutil.copytree(DRIVE_DATASET_ROOT, LOCAL_DATASET_ROOT)
        copy_time_min = (time.time() - copy_start) / 60.0
        print(f"Dataset copy completed in {copy_time_min:.2f} minutes.")

    DATASET_ROOT = LOCAL_DATASET_ROOT
else:
    DATASET_ROOT = DRIVE_DATASET_ROOT

train_good_path = DATASET_ROOT / "nondefects" / "nondefects"
test_root_path  = DATASET_ROOT / "test" / "test"

if not train_good_path.exists():
    raise ValueError(f"Training path not found: {train_good_path}")

if not test_root_path.exists():
    raise ValueError(f"Test path not found: {test_root_path}")

print("Using DATASET_ROOT:", DATASET_ROOT)
print("Train folder:", train_good_path)
print("Test folder :", test_root_path)

# ============================================================
# 4) Fixed PatchCore settings
# Keep these fixed for fair layer comparison
# ============================================================

BACKBONE_NAME = "EfficientNet-B5"

IMG_SIZE = 448
BATCH_SIZE = 16
NUM_WORKERS = 0

PATCHES_PER_IMAGE = 200
PRE_POOL = 400_000
MAX_MEM_PATCHES = 20_000

NN_CHUNK = 40_000
CORESET_CHUNK = 40_000
THRESH_SAMPLE_IMAGES = 2000

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE != "cuda":
    raise RuntimeError("Please enable GPU in Colab: Runtime > Change runtime type > GPU")

print("Using device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 5) Layer combinations to test
# ============================================================

LAYER_CONFIGS = [
    {
        "layers": [2],
        "type": "Early texture",
    },
    {
        "layers": [3],
        "type": "Early-mid texture",
    },
    {
        "layers": [5],
        "type": "Mid-deep structure",
    },
    {
        "layers": [7],
        "type": "Deep semantic",
    },
    {
        "layers": [2, 3],
        "type": "Early fusion",
    },
    {
        "layers": [3, 5],
        "type": "Mid fusion",
    },
    {
        "layers": [5, 7],
        "type": "Deep fusion",
    },
    {
        "layers": [2, 3, 5],
        "type": "Texture + structure fusion",
    },
    {
        "layers": [3, 5, 7],
        "type": "Current baseline",
    },
    {
        "layers": [2, 3, 5, 7],
        "type": "Full multi-scale fusion",
    },
]

# ============================================================
# 6) Output directory
# ============================================================

SAVE_DIR = Path("/content/drive/MyDrive/<YOUR_OUTPUT_FOLDER>/layer_combination_comparison")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

RESULT_CSV = SAVE_DIR / "efficientnet_b5_patchcore_greedy_layer_comparison_local_timing_seed42.csv"

print("Results will be saved to:", RESULT_CSV)

# ============================================================
# 7) Size and memory helpers
# ============================================================

def bytes_to_mb(x):
    return x / (1024 ** 2)

def tensor_size_mb(tensor):
    return bytes_to_mb(tensor.numel() * tensor.element_size())

def model_size_mb(model):
    total_bytes = 0

    for p in model.parameters():
        total_bytes += p.numel() * p.element_size()

    for b in model.buffers():
        total_bytes += b.numel() * b.element_size()

    return bytes_to_mb(total_bytes)

def reset_peak_gpu_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

def peak_gpu_memory_mb():
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        return bytes_to_mb(torch.cuda.max_memory_allocated())
    return 0.0

def edge_efficiency_score(auc, ap, f1, time_per_image, total_mb, peak_gpu_mb):
    """
    Weighted raw edge-efficiency score.

    E = (0.25*AUC + 0.35*AP + 0.40*F1)
        /
        (0.20*T + 0.40*M + 0.40*G)

    T = inference time per image in seconds
    M = total footprint in MB
    G = peak GPU memory in MB

    Higher is better.
    """
    numerator = 0.25 * auc + 0.35 * ap + 0.40 * f1
    denominator = 0.20 * time_per_image + 0.40 * total_mb + 0.40 * peak_gpu_mb

    if denominator <= 0:
        return 0.0

    return numerator / denominator

# ============================================================
# 8) Transform
# Important: No CenterCrop
# ============================================================

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# ============================================================
# 9) Dataset utilities
# ============================================================

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def list_images(folder: Path, recursive=True):
    paths = []
    iterator = folder.rglob("*") if recursive else folder.glob("*")

    for p in sorted(iterator):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            try:
                with Image.open(p) as img:
                    img.verify()
                paths.append(p)
            except Exception as e:
                print("Corrupted image skipped:", p, e)

    return paths

class ImagePathDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = list(paths)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]
        img = Image.open(p).convert("RGB")
        return self.transform(img), str(p)

class TestImageDataset(Dataset):
    def __init__(self, items, transform):
        self.items = list(items)
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        p, label = self.items[idx]
        img = Image.open(p).convert("RGB")
        return self.transform(img), int(label), str(p)

# ============================================================
# 10) Load train/test paths once
# ============================================================

train_paths = list_images(train_good_path)

if len(train_paths) == 0:
    raise ValueError("No training normal images found.")

print("Training normal images:", len(train_paths))

def get_label(folder_name):
    name = folder_name.lower().replace("_", "-").strip()

    if name == "non-defects":
        return 0

    if name == "defects":
        return 1

    return None

test_items = []

for folder in sorted(test_root_path.iterdir()):
    if not folder.is_dir():
        continue

    label = get_label(folder.name)

    if label is None:
        print("Skipping unknown folder:", folder.name)
        continue

    paths = list_images(folder)

    for p in paths:
        test_items.append((p, label))

if len(test_items) == 0:
    raise ValueError("No test images found.")

print("Total test images:", len(test_items))
print("Normal test images:", sum(1 for _, y in test_items if y == 0))
print("Defect test images:", sum(1 for _, y in test_items if y == 1))

train_loader = DataLoader(
    ImagePathDataset(train_paths, transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
    drop_last=False,
)

test_loader = DataLoader(
    TestImageDataset(test_items, transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
    drop_last=False,
)

# ============================================================
# 11) Backbone feature extractor
# ============================================================

class EfficientNetB5FeatureExtractor(torch.nn.Module):
    def __init__(self, layer_indices):
        super().__init__()

        self.layer_indices = list(layer_indices)

        self.model = efficientnet_b5(weights=EfficientNet_B5_Weights.DEFAULT)
        self.model.eval()

        for p in self.model.parameters():
            p.requires_grad = False

        self.features = []

        def hook(_, __, output):
            self.features.append(output)

        for idx in self.layer_indices:
            self.model.features[idx].register_forward_hook(hook)

    @torch.no_grad()
    def forward(self, x):
        self.features = []

        _ = self.model(x)

        if len(self.features) == 0:
            raise RuntimeError("No features extracted. Check hook registration.")

        # Resize all selected feature maps to the smallest spatial resolution
        fmap_size = min(f.shape[-2] for f in self.features)
        resize = torch.nn.AdaptiveAvgPool2d(fmap_size)

        resized = [resize(f) for f in self.features]
        patch_features = torch.cat(resized, dim=1)

        B, C, H, W = patch_features.shape
        patch_features = patch_features.reshape(B, C, H * W).permute(0, 2, 1)

        return patch_features  # (B, N, D)

# ============================================================
# 12) Greedy Coreset
# ============================================================

@torch.no_grad()
def greedy_coreset_gpu(
    features_cpu,
    max_samples,
    chunk=40_000,
    use_fp16=True,
    device="cuda",
    seed=42,
):
    random.seed(seed)

    N, C = features_cpu.shape

    if N <= max_samples:
        return features_cpu.clone()

    feats = features_cpu.to(device, non_blocking=True).contiguous()

    if use_fp16:
        feats = feats.half()

    selected_idx = torch.empty((max_samples,), dtype=torch.long, device=device)

    first = random.randint(0, N - 1)
    selected_idx[0] = first

    center = feats[first:first + 1]

    min_d = torch.empty((N,), device=device, dtype=torch.float32)

    for start in range(0, N, chunk):
        x = feats[start:start + chunk]
        d = (x - center).float().pow(2).sum(dim=1)
        min_d[start:start + chunk] = d

    for i in tqdm(range(1, max_samples), desc="Greedy coreset"):
        farthest = torch.argmax(min_d).item()
        selected_idx[i] = farthest

        center = feats[farthest:farthest + 1]

        for start in range(0, N, chunk):
            x = feats[start:start + chunk]
            d = (x - center).float().pow(2).sum(dim=1)
            min_d[start:start + chunk] = torch.minimum(
                min_d[start:start + chunk],
                d,
            )

    selected = feats[selected_idx].float().cpu()

    del feats, selected_idx, min_d

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return selected

# ============================================================
# 13) PatchCore image score
# ============================================================

@torch.no_grad()
def image_anomaly_score(patch_feats_gpu, memory_bank_gpu, chunk_size=40_000):
    x = patch_feats_gpu.float()
    P = x.shape[0]

    min_dist = torch.full(
        (P,),
        float("inf"),
        device=DEVICE,
        dtype=torch.float32,
    )

    x2 = x.pow(2).sum(dim=1, keepdim=True)

    for start in range(0, memory_bank_gpu.shape[0], chunk_size):
        mb = memory_bank_gpu[start:start + chunk_size].float()
        mb2 = mb.pow(2).sum(dim=1).unsqueeze(0)

        dot = x @ mb.t()
        d2 = x2 + mb2 - 2.0 * dot
        d2 = torch.clamp(d2, min=0.0)

        min_dist = torch.minimum(min_dist, d2.min(dim=1).values)

    # Max image score
    return min_dist.sqrt().max().item()

# ============================================================
# 14) Load previous results if available
# ============================================================

if RESULT_CSV.exists():
    df_results = pd.read_csv(RESULT_CSV)
    print("\nExisting result file found. Completed rows:")
    display(df_results)
else:
    df_results = pd.DataFrame()

completed_keys = set()

if not df_results.empty:
    completed_keys = set(df_results["Layer_Combination"].astype(str).tolist())

# ============================================================
# 15) Single layer-combination run function
# ============================================================

def run_one_layer_config(layer_indices, layer_type):
    layer_name = ",".join([str(i) for i in layer_indices])
    layer_display = "features[" + "], features[".join([str(i) for i in layer_indices]) + "]"

    print("\n" + "=" * 80)
    print("Running layer combination:", layer_display)
    print("Layer type:", layer_type)
    print("=" * 80)

    set_seed(SEED)
    reset_peak_gpu_memory()

    # ----------------------------
    # Build backbone
    # ----------------------------
    backbone = EfficientNetB5FeatureExtractor(layer_indices).to(DEVICE).eval()
    backbone_size_mb = model_size_mb(backbone)

    with torch.no_grad():
        dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
        dummy_feats = backbone(dummy)

    feature_dim_d = int(dummy_feats.shape[-1])
    patch_grid_n = int(dummy_feats.shape[1])

    print("Feature dimension D:", feature_dim_d)
    print("Patch grid count N:", patch_grid_n)
    print(f"Backbone size MB: {backbone_size_mb:.2f}")

    del dummy, dummy_feats

    # ----------------------------
    # Build candidate patch pool
    # ----------------------------
    all_features = []

    print("\nExtracting candidate patch pool...")

    for xb, _ in tqdm(train_loader):
        xb = xb.to(DEVICE, non_blocking=True)

        with torch.no_grad():
            feats = backbone(xb)

        B, N, D = feats.shape

        for i in range(B):
            n = min(PATCHES_PER_IMAGE, N)
            idx = torch.randperm(N, device=DEVICE)[:n]
            sampled = feats[i, idx].detach().float().cpu()
            all_features.append(sampled)

        del xb, feats

    candidate_pool = torch.cat(all_features, dim=0)
    del all_features

    print("Candidate pool before PRE_POOL cap:", tuple(candidate_pool.shape))

    if candidate_pool.shape[0] > PRE_POOL:
        set_seed(SEED)
        idx = torch.randperm(candidate_pool.shape[0])[:PRE_POOL]
        candidate_pool = candidate_pool[idx]

    print("Final candidate pool:", tuple(candidate_pool.shape))

    # ----------------------------
    # Greedy Coreset memory bank
    # ----------------------------
    print("\nBuilding Greedy Coreset memory bank...")

    memory_bank_cpu = greedy_coreset_gpu(
        candidate_pool,
        max_samples=MAX_MEM_PATCHES,
        chunk=CORESET_CHUNK,
        use_fp16=(DEVICE == "cuda"),
        device=DEVICE,
        seed=SEED,
    )

    del candidate_pool

    memory_bank_size_mb = tensor_size_mb(memory_bank_cpu)
    total_footprint_mb = backbone_size_mb + memory_bank_size_mb

    print(f"Memory bank size MB: {memory_bank_size_mb:.2f}")
    print(f"Estimated total footprint MB: {total_footprint_mb:.2f}")

    memory_bank_gpu = memory_bank_cpu.to(DEVICE, non_blocking=True)

    # ----------------------------
    # Threshold calculation
    # ----------------------------
    rng = np.random.default_rng(SEED + 100)

    num_for_thresh = min(THRESH_SAMPLE_IMAGES, len(train_paths))
    thresh_indices = rng.permutation(len(train_paths))[:num_for_thresh]
    thresh_subset = [train_paths[i] for i in thresh_indices]

    thresh_loader = DataLoader(
        ImagePathDataset(thresh_subset, transform),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        drop_last=False,
    )

    threshold_scores = []

    print("\nComputing threshold from normal training subset...")

    for xb, _ in tqdm(thresh_loader, desc="Threshold"):
        xb = xb.to(DEVICE, non_blocking=True)

        with torch.no_grad():
            feats = backbone(xb)

        for i in range(feats.shape[0]):
            score = image_anomaly_score(
                feats[i],
                memory_bank_gpu,
                chunk_size=NN_CHUNK,
            )
            threshold_scores.append(score)

        del xb, feats

    threshold_scores = np.array(threshold_scores)
    threshold = threshold_scores.mean() + 3.0 * threshold_scores.std(ddof=1)

    print(f"Internal threshold for F1: {threshold:.6f}")

    # ----------------------------
    # Test evaluation
    # ----------------------------
    # compute_total_time measures only batch transfer + model feature extraction + NN scoring.
    # end_to_end_total_time additionally includes DataLoader/PIL/transform and loop overhead.
    reset_peak_gpu_memory()

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    end_to_end_start = time.perf_counter()
    compute_total_time = 0.0

    y_true = []
    y_score = []

    print("\nEvaluating test set...")

    for xb, yb, _ in tqdm(test_loader, desc="Testing"):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        compute_start = time.perf_counter()

        xb = xb.to(DEVICE, non_blocking=True)

        with torch.no_grad():
            feats = backbone(xb)

        for i in range(feats.shape[0]):
            score = image_anomaly_score(
                feats[i],
                memory_bank_gpu,
                chunk_size=NN_CHUNK,
            )

            y_score.append(score)
            y_true.append(int(yb[i]))

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        compute_total_time += time.perf_counter() - compute_start

        del xb, feats

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    end_to_end_total_time = time.perf_counter() - end_to_end_start
    compute_time_per_image = compute_total_time / len(y_true)
    end_to_end_time_per_image = end_to_end_total_time / len(y_true)
    peak_gpu_memory = peak_gpu_memory_mb()

    # ----------------------------
    # Metrics
    # ----------------------------
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    y_pred = (y_score > threshold).astype(int)

    auc_roc = roc_auc_score(y_true, y_score)
    ap = average_precision_score(y_true, y_score)
    f1 = f1_score(y_true, y_pred)

    edge_eff = edge_efficiency_score(
        auc=auc_roc,
        ap=ap,
        f1=f1,
        time_per_image=compute_time_per_image,
        total_mb=total_footprint_mb,
        peak_gpu_mb=peak_gpu_memory,
    )

    result = {
        "Dataset_Root_Used": str(DATASET_ROOT),
        "Backbone": BACKBONE_NAME,
        "Layer_Combination": layer_display,
        "Layer_Key": layer_name,
        "Layer_Type": layer_type,
        "D": feature_dim_d,
        "Patch_Grid_N": patch_grid_n,
        "Backbone_MB": backbone_size_mb,
        "Memory_Bank_MB": memory_bank_size_mb,
        "Total_MB": total_footprint_mb,
        "Peak_GPU_Memory_MB": peak_gpu_memory,
        "Time/Image": compute_time_per_image,
        "Compute_Inference_Time_Per_Image_sec": compute_time_per_image,
        "End_To_End_Local_Runtime_Per_Image_sec": end_to_end_time_per_image,
        "Compute_Total_Time_sec": compute_total_time,
        "End_To_End_Local_Total_Time_sec": end_to_end_total_time,
        "AUC": auc_roc,
        "AP": ap,
        "F1": f1,
        "Internal_Threshold": threshold,
        "Edge_Efficiency": edge_eff,
    }

    # Cleanup
    del backbone, memory_bank_cpu, memory_bank_gpu
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

# ============================================================
# 16) Run all layer combinations one by one
# ============================================================

for cfg in LAYER_CONFIGS:
    layer_indices = cfg["layers"]
    layer_type = cfg["type"]

    layer_display = "features[" + "], features[".join([str(i) for i in layer_indices]) + "]"

    if layer_display in completed_keys:
        print(f"\nSkipping already completed: {layer_display}")
        continue

    result = run_one_layer_config(layer_indices, layer_type)

    df_new = pd.DataFrame([result])

    if RESULT_CSV.exists():
        df_old = pd.read_csv(RESULT_CSV)
        df_all = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df_all = df_new

    # Compute normalized edge efficiency percentage after each new row
    max_eff = df_all["Edge_Efficiency"].max()

    if max_eff > 0:
        df_all["Edge_Efficiency_%"] = (df_all["Edge_Efficiency"] / max_eff) * 100.0
    else:
        df_all["Edge_Efficiency_%"] = 0.0

    df_all.to_csv(RESULT_CSV, index=False)

    print("\nUpdated results saved to:")
    print(RESULT_CSV)

    display(df_all)

# ============================================================
# 17) Final display sorted by AP, F1, and edge efficiency
# ============================================================

df_final = pd.read_csv(RESULT_CSV)

max_eff = df_final["Edge_Efficiency"].max()

if max_eff > 0:
    df_final["Edge_Efficiency_%"] = (df_final["Edge_Efficiency"] / max_eff) * 100.0
else:
    df_final["Edge_Efficiency_%"] = 0.0

df_final.to_csv(RESULT_CSV, index=False)

print("\n" + "=" * 80)
print("FINAL LAYER COMBINATION RESULTS")
print("=" * 80)

display(df_final)

print("\nSorted by AP:")
display(df_final.sort_values("AP", ascending=False))

print("\nSorted by F1:")
display(df_final.sort_values("F1", ascending=False))

print("\nSorted by Edge Efficiency:")
display(df_final.sort_values("Edge_Efficiency", ascending=False))

print("\nBest by AP:")
display(df_final.sort_values("AP", ascending=False).head(1))

print("\nBest by F1:")
display(df_final.sort_values("F1", ascending=False).head(1))

print("\nBest by Edge Efficiency:")
display(df_final.sort_values("Edge_Efficiency", ascending=False).head(1))